In [2]:
%pip install flask pandas

import sqlite3
import pandas as pd
from flask import Flask, jsonify

transactions = pd.DataFrame({
    "transaction_id": [1, 2, 3, 4, 5],
    "type": ["deposit", "withdrawal", "deposit",
             "withdrawal", "deposit"],
    "amount": [500, 100, 250, 50, 300],
    "status": ["completed", "completed", "completed",
               "failed", "completed"]
})

transactions

Note: you may need to restart the kernel to use updated packages.


,transaction_id,type,amount,status
0,1,deposit,500,completed
1,2,withdrawal,100,completed
2,3,deposit,250,completed
3,4,withdrawal,50,failed
4,5,deposit,300,completed


In [4]:
conn = sqlite3.connect("banking.db")

transactions.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

pd.read_sql_query(
    "SELECT * FROM transactions",
    conn
)

,transaction_id,type,amount,status
0,1,deposit,500,completed
1,2,withdrawal,100,completed
2,3,deposit,250,completed
3,4,withdrawal,50,failed
4,5,deposit,300,completed


In [6]:
app = Flask(__name__)

@app.route("/transactions", methods=["GET"])
def get_transactions():
    conn = sqlite3.connect("banking.db")
    df = pd.read_sql_query(
        "SELECT * FROM transactions",
        conn
    )
    conn.close()

    return jsonify(df.to_dict(orient="records"))


@app.route("/summary", methods=["GET"])
def get_summary():
    conn = sqlite3.connect("banking.db")

    df = pd.read_sql_query(
        "SELECT * FROM transactions",
        conn
    )
    conn.close()

    completed = df[df["status"] == "completed"]

    deposits = completed.loc[
        completed["type"] == "deposit", "amount"
    ].sum()

    withdrawals = completed.loc[
        completed["type"] == "withdrawal", "amount"
    ].sum()

    return jsonify({
        "total_deposits": float(deposits),
        "total_withdrawals": float(withdrawals),
        "net_balance_change": float(deposits - withdrawals),
        "completed_transactions": int(len(completed))
    })

In [8]:
client = app.test_client()

response = client.get("/summary")

print(response.status_code)
print(response.json)

200
{'completed_transactions': 4, 'net_balance_change': 950.0, 'total_deposits': 1050.0, 'total_withdrawals': 100.0}


In [10]:
response = client.get("/transactions")

assert response.status_code == 200
assert len(response.json) == 5

print("All tests passed!")

All tests passed!


In [12]:
if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
On macOS, try disabling the 'AirPlay Receiver' service from System Preferences -> General -> AirDrop & Handoff.


SystemExit: 1

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
